## Case Study: Debugging a Multi-Layer Production Agent Implementation

**1. Context & Architecture Overview**

A developer implemented an AI agent loop designed to handle customer data queries using Claude with tool calling, extended thinking, streaming responses, and session memory. However, during initial load and edge-case testing, four critical failure modes were identified across four distinct architectural layers.

In [ ]:
# 2. The Buggy Implementation
# --- TOOL DEFINITIONS ---
tools = [
    {
        "name": "get_customer_data",
        "description": "Gets data.",  # Bug 1
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"],
        },
    }
]


# --- AGENT LOOP ---
def run_agent(user_request, session_history):
    messages = session_history + [{"role": "user", "content": user_request}]

    while True:
        blocks = {}
        stop_seen = False

        with client.messages.stream(
            model=model,
            max_tokens=4096,
            tools=tools,
            messages=messages,
            thinking={"type": "adaptive"},
        ) as stream:
            for event in stream:
                if event.type == "content_block_start":
                    blocks[event.index] = init_block(event)
                elif event.type == "content_block_delta":
                    apply_delta(blocks[event.index], event.delta)
                elif event.type == "message_stop":
                    stop_seen = True

        # Bug 2 & Bug 3
        assistant_content = [
            b for b in assemble(blocks) if b["type"] != "thinking"
        ]
        messages.append({"role": "assistant", "content": assistant_content})

        response = finalize(blocks)
        if response.stop_reason == "end_turn":
            return response

        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                messages.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": result,
                            }
                        ],
                    }
                )


# --- MEMORY ---
def build_session_history(prior_sessions):
    # Bug 4
    full_history = []
    for session in prior_sessions:
        full_history.extend(session["messages"])
    return full_history

**3. Failure Mode Analysis & Solutions**

Defect 1: Schema Layer (Vague Tool Description)
- Root Cause: The description "Gets data." lacks intent, target usage boundaries, and domain restrictions. Claude cannot reliably determine when to trigger the tool versus when to use an alternate function.

- Impact: High tool routing misfires and hallucinations when users ask for order history or transactions.

- Fix: Add clear functional boundaries specifying intended usage and explicit exclusions.

In [ ]:
# Fixed Description
"description": "Use this to retrieve full account and contact details for a customer by customer ID. Do not use this for order history or transaction records."

**Defect 2 & 3: Streaming & Context Layer (Thinking Block Stripping & Un-gated Commit)**

Root Cause:

   1. The code strips thinking blocks (b["type"] != "thinking") before appending the assistant's turn to messages. Modifying or omitting thinking blocks invalidates signature validation on subsequent turns.

   2. messages.append(...) is called unconditionally outside the stream loop, ignoring the stop_seen status and risking partial message commits if the stream drops mid-way.

- Impact: Session failure on multi-turn tool calling and corrupted conversation history on stream timeouts.

- Fix: Retain all content blocks (including thinking), gate state updates behind stop_seen, and raise an exception if interrupted.

In [ ]:
# Fixed Streaming & Context Assembly

assistant_content = assemble(blocks)  # Retain all blocks including 'thinking'

if stop_seen:
    messages.append({"role": "assistant", "content": assistant_content})
else:
    raise StreamInterruptedError(
        "Discarding partial turn; retry from last complete turn."
    )

**Defect 4: Memory Layer (In-Context History Inflation)**

- Root Cause: build_session_history concatenates full raw message transcripts across all prior sessions into the prompt context window.

- Impact: Excessive token consumption, high latency, scaling cost issues, and risk of exceeding context limits.

- Fix: Store session logs externally and inject a single summary at session start.

In [ ]:
# Fixed External Storage / Summary Injection
def build_session_history(prior_sessions):
    if not prior_sessions:
        return []

    # Fetch summary from external database instead of raw transcripts
    summary = load_session_summary(prior_sessions[-1]["id"])
    return [{"role": "user", "content": f"Session context: {summary}"}]

**4. Key Takeaways**

1. _Schema Specificity_: Always specify what a tool does and what it does not do.

2. _State Protection_: Never strip or alter signature-validated blocks (thinking) across multi-turn sessions, and only commit turns to history when message_stop guarantees completeness.

3. _Context Hygiene_: Use external storage and summarization rather than re-playing raw conversational logs across sessions.